# Notebook 12 — Middleware + Context Management for the Research Deep Agent

This notebook continues the `deep-agents-on-foundry` learning journey.

By now, the research agent already has:
- streaming
- threads/checkpoint persistence
- native durability/HITL
- Hosted Agent integration
- memory
- skills

The next question is:

> **What should the model actually see on each call?**

That is the heart of **context management**.

Middleware is the mechanism that can shape model requests, tool availability,
message history, memory injection, skill exposure, summarization, and other
run-time behaviors around the model call.

## Learning goals

By the end, you should understand:

1. What middleware is
2. Middleware vs tools
3. The Deep Agents middleware stack
4. Context-window pressure and context bloat
5. Automatic summarization
6. History offloading
7. Tool-argument truncation
8. Manual/agent-triggered compaction
9. Dynamic tool filtering
10. Middleware ordering and composition
11. When custom middleware is justified
12. How to compare context strategies
13. How tracing/evals relate to context management
14. Why context engineering is an economics problem

# 12.1 — What is middleware?

The simplest intuition:

```text
Tool:
the model chooses to call me

Middleware:
I can modify what happens around the model call
```

Tools execute actions.

Middleware can shape:
- the request before the model sees it
- the tools exposed to the model
- the messages included in context
- the system instructions
- how history is compacted
- how memory/skills are injected
- how execution state is managed around calls

Mental model:

```text
User message
   ↓
LangGraph state
   ↓
Middleware stack
   ├── memory injection
   ├── skills catalog
   ├── tool filtering
   ├── summarization
   ├── filesystem/context offload
   └── request shaping
   ↓
Model call
```

# 12.2 — Middleware vs tools

A tool is a primitive capability:

```text
web_search(query)
read_file(path)
write_file(path, content)
```

Middleware is more like a policy layer:

```text
before model call:
- hide irrelevant tools
- inject relevant memory
- expose skill metadata
- summarize old messages
- trim large tool payloads
```

A useful rule:

> Use a tool for an action.
> Use middleware for behavior that must shape or intercept the model/runtime flow.

# 12.3 — Inspect the installed middleware APIs

Deep Agents and LangGraph evolve quickly, so inspect the installed package rather than
assuming exact signatures from another version.

In [ ]:
import inspect

from deepagents import create_deep_agent

print("create_deep_agent:")
print(inspect.signature(create_deep_agent))

In [ ]:
# Inspect middleware modules that exist in your installed version.

try:
    import deepagents.middleware as dam
    print("deepagents.middleware members:")
    print(sorted(name for name in dir(dam) if not name.startswith("_")))
except Exception as exc:
    print("Could not inspect deepagents.middleware:", exc)

# 12.4 — Context-window pressure

A research agent naturally accumulates context:

```text
user question
  ↓
web search result
  ↓
model response
  ↓
more search
  ↓
tool output
  ↓
follow-up
  ↓
more research
```

Without context management:

```text
history grows
   ↓
input tokens grow
   ↓
latency grows
   ↓
cost grows
   ↓
signal-to-noise can worsen
   ↓
eventually context limit is reached
```

The important point:

> Context management is not only about avoiding the hard context-window limit.

Even before the limit, too much context can reduce quality because the model has to
reason over more irrelevant material.

## 12.4A — Context quality > context quantity

A better model of agent performance is:

```text
model quality
× context quality
× tool quality
× orchestration quality
```

A larger context is not automatically a better context.

Useful context management tries to maximize:

```text
relevant signal
----------------
total context
```

# 12.5 — Summarization as context compaction

One of the most important context-management mechanisms is summarization.

Conceptually:

```text
old detailed history
        ↓
summarization
        ↓
compact summary
        ↓
active model context
```

But good agent summarization should ideally preserve the ability to inspect or recover
the original history elsewhere.

So a stronger architecture is:

```text
active context
    ↓ compact
summary remains hot

old full history
    ↓
backend/file storage
```

This resembles a systems memory hierarchy:

```text
L1: active prompt context
L2: compact summary
L3: recoverable history/artifacts
```

# 12.6 — Inspect Deep Agents summarization support

Try to inspect the installed summarization middleware APIs.

In [ ]:
try:
    from deepagents.middleware import summarization as summarization_module
    print(sorted(name for name in dir(summarization_module) if not name.startswith("_")))
except Exception as exc:
    print("Summarization module inspection failed:", exc)

In [ ]:
# Inspect likely summarization constructors/classes if present.

candidates = [
    "SummarizationMiddleware",
    "SummarizationToolMiddleware",
    "create_summarization_middleware",
    "create_summarization_tool_middleware",
]

if 'summarization_module' in globals():
    for name in candidates:
        obj = getattr(summarization_module, name, None)
        if obj is not None:
            try:
                print(name, "->", inspect.signature(obj))
            except Exception:
                print(name, "->", obj)

# 12.7 — Automatic summarization

Automatic summarization typically uses a threshold policy.

Example mental model:

```text
message history grows
        ↓
token threshold reached
        ↓
summarization middleware
        ↓
older content summarized
        ↓
recent/high-value context retained
```

Two questions matter:

1. **When should compaction trigger?**
2. **How much recent detail should remain uncompressed?**

Common trigger styles:
- absolute token threshold
- fraction of model context window
- message count
- explicit overflow recovery

Common keep styles:
- last N messages
- last X% of context
- important recent tool results

## 12.7A — Why aggressive thresholds are useful in a notebook

In a real deployment, summarization might trigger near a large context budget.

For learning, use an intentionally low threshold so you can observe the behavior
without generating tens of thousands of tokens.

The goal is to understand the mechanism, not benchmark production thresholds yet.

In [ ]:
# Conceptual example only.
# Adapt to the exact installed API discovered above.

# summarization = create_summarization_middleware(
#     model=model,
#     backend=backend,
#     trigger=("fraction", 0.30),
#     keep=("fraction", 0.10),
# )

# 12.8 — History offloading

Good compaction should avoid destructive forgetting when possible.

Conceptually:

```text
before:
[full old history] + [recent history]

after:
[summary] + [recent history]

cold storage:
[full evicted history]
```

Why this matters:
- traceability
- debugging
- evaluation
- recovering details
- auditability
- future rehydration if necessary

For your project, this is especially important because you care about
trace/eval-driven improvement and token economics.

# 12.9 — Tool-argument truncation

Context bloat does not come only from conversation messages.

Large tool arguments/results can also dominate context.

Example:

```text
write_file(
    path="report.md",
    content="<10,000-token report>"
)
```

If the full argument is repeatedly replayed into future model calls, that can become
very expensive.

A context-management layer may:
- retain the semantic fact that the file was written
- store the full content externally
- keep only a compact reference in active context

Mental model:

```text
huge tool payload
   ↓
offload
   ↓
small reference in context
```

# 12.10 — Manual / agent-triggered compaction

There are two broad compaction models:

```text
automatic compaction
→ runtime decides from thresholds

agent-triggered compaction
→ model calls a compact/summarize tool
```

Automatic compaction is predictable.

Agent-triggered compaction can be adaptive, but relies on the model deciding correctly.

A hybrid is often reasonable:
- agent may compact when useful
- runtime still has a safety threshold

In [ ]:
# Inspect whether a compact-conversation or summarization tool middleware exists
# in your installed version.

if 'summarization_module' in globals():
    for name in dir(summarization_module):
        if "compact" in name.lower() or "tool" in name.lower():
            print(name)

# 12.11 — Build a focused summarization experiment

The preferred experiment for this notebook:

```text
same research task
    ↓
baseline agent
vs
summarization-enabled agent
```

Measure:
- input tokens
- output tokens
- latency
- model-call count
- final quality
- facts preserved
- important facts lost

Do not conclude "summarization is better" simply because the prompt got shorter.

In [ ]:
from deep_agents_foundry.agent import RESEARCH_INSTRUCTIONS
from deep_agents_foundry.model import build_model
from deep_agents_foundry.tools import build_web_search_tool
from deep_agents_foundry import (
    build_sqlite_checkpointer,
    thread_config,
    content_text,
)

model = build_model()
web_search = build_web_search_tool()
checkpointer = build_sqlite_checkpointer("../data/checkpoints.db")

## 12.11A — Baseline agent

Use the normal Deep Agent first.

In [ ]:
baseline_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=RESEARCH_INSTRUCTIONS,
    checkpointer=checkpointer,
)

Use a multi-turn task that naturally accumulates history.

For example:

```text
Turn 1:
Research Foundry Hosted Agents architecture.

Turn 2:
Now focus on identity and RBAC.

Turn 3:
Compare its state/session model with LangGraph threads.

Turn 4:
Summarize the most important trade-offs.
```

In [ ]:
baseline_thread = thread_config("context-baseline-1")

# Example first turn:
#
# result = baseline_agent.invoke(
#     {"messages": [{
#         "role": "user",
#         "content": "Research Microsoft Foundry Hosted Agents architecture."
#     }]},
#     config=baseline_thread,
# )
# print(content_text(result["messages"][-1]))

## 12.11B — Summarization-enabled agent

After inspecting the exact installed summarization API, create a second agent with
aggressive learning-friendly thresholds.

Conceptually:

```python
summarizing_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=RESEARCH_INSTRUCTIONS,
    middleware=[summarization],
    checkpointer=checkpointer,
    backend=backend,
)
```

Use a different thread ID so baseline and summarized runs remain isolated.

In [ ]:
# Build this only after adapting to the exact installed middleware/backend signatures.
#
# summarizing_agent = create_deep_agent(
#     model=model,
#     tools=[web_search],
#     system_prompt=RESEARCH_INSTRUCTIONS,
#     middleware=[summarization],
#     checkpointer=checkpointer,
#     backend=backend,
# )

# 12.12 — Dynamic tool filtering

One of the strongest uses of middleware is controlling which tools the model sees.

This matters because every tool adds:
- schema/context overhead
- routing choices
- opportunities for wrong tool use

You already observed earlier in this project that merely adding Web Search changed agent
behavior dramatically.

Middleware can potentially implement:

```text
model call A
tools = [web_search, read_file]

model call B
tools = [read_file]

model call C
tools = [web_search]
```

That is stronger than giving the model every tool on every turn and relying only on
prompt instructions.

## 12.12A — Tool filtering as context management

Tool schemas are context too.

So context management includes:

```text
messages
+
memory
+
skills
+
files
+
tool schemas
```

This means tool routing has both:
- reasoning implications
- token economics implications

# 12.13 — Middleware ordering and composition

Middleware can interact.

Suppose you eventually have:

```text
MemoryMiddleware
SkillsMiddleware
SummarizationMiddleware
ToolFilterMiddleware
```

Possible order:

```text
retrieve memory
   ↓
expose skills
   ↓
compact history
   ↓
filter tools
   ↓
model call
```

Different hook ordering may change the resulting request.

General rule:

> Middleware is composable, but compositions are not automatically independent.

Whenever multiple middleware components are added, validate the combined behavior.

# 12.14 — When should you write custom middleware?

Write custom middleware when behavior must shape or intercept model/runtime execution.

Good candidates:
- dynamic tool filtering
- context-budget policy
- research token/search budget
- runtime-specific system instructions
- source-policy injection
- request shaping

Do NOT write custom middleware for a simple stateless action.

That should remain a normal tool.

## 12.14A — Possible future middleware for this research agent

Potential later ideas:

```text
ResearchBudgetMiddleware
→ constrain search/token budget

SourcePolicyMiddleware
→ prefer/require authoritative sources under certain tasks

ToolRoutingMiddleware
→ hide irrelevant tools dynamically

ContextBudgetMiddleware
→ adapt retained context by task complexity
```

Do not implement these yet.

First prove the built-in context-management behavior and measure it.

# 12.15 — Context assembly hierarchy

At this point, the model context may be assembled from:

```text
system prompt
+
memory files
+
retrieved long-term memories
+
selected skill bodies
+
thread history
+
summaries
+
needed files/artifacts
+
available tool schemas
```

That is the real context-engineering layer.

The key question becomes:

> What information does the model need **for this decision right now**?

# 12.16 — Trace + eval implications

Context management should be observable.

Useful trace questions:
- How many model calls happened?
- What tool calls happened?
- How large was input context over time?
- When did summarization trigger?
- Which memories/skills were injected?
- Which tools were exposed?
- Did compaction cause repeated searches?
- Did the agent lose an important fact after summarization?

Useful eval questions:
- Was the final answer still correct?
- Were critical facts preserved?
- Did the agent repeat work?
- Did tool selection improve?
- Did cost fall without quality loss?

# 12.17 — Economics

Context management is directly an economics problem.

Illustrative example:

```text
without compaction:
call 1 = 5K input
call 2 = 10K
call 3 = 20K
call 4 = 35K
total = 70K

with compaction:
call 1 = 5K
call 2 = 10K
compact
call 3 = 8K
call 4 = 12K
total = 35K
```

But summarization itself has cost and can lose information.

So the real question is:

> Does compaction reduce enough future work while preserving enough quality?

That is exactly the kind of trade-off the later agent-economics notebook should measure.

# 12.18 — Before/after experiment template

Use this table after running the baseline and summarizing agents:

| Metric | Baseline | Summarized |
|---|---:|---:|
| Input tokens | | |
| Output tokens | | |
| Model calls | | |
| Web searches | | |
| Latency | | |
| Quality score | | |
| Critical facts preserved | | |
| Repeated work | | |

Do not optimize only for token reduction.

The target is better **value per unit of context**.

# 12.19 — Context-management decision framework

For each context item, ask:

1. Does the model need this now?
2. Is it stable or time-sensitive?
3. Can it be retrieved later?
4. Can it be summarized safely?
5. Can it be offloaded to a file/store?
6. Does it need to be in every model call?
7. What is the cost of keeping it?
8. What is the risk of removing it?

This is the practical heart of context engineering.

# Notebook 12 — Key takeaways

1. Middleware shapes behavior around model/runtime execution.
2. Tools perform actions; middleware shapes context/policy.
3. Context bloat hurts cost, latency, and sometimes reasoning quality.
4. Summarization is a form of context compaction.
5. Good systems offload old detail rather than blindly delete it.
6. Tool payloads and tool schemas are part of the context budget.
7. Dynamic tool filtering can reduce routing mistakes and token overhead.
8. Middleware ordering can create interactions.
9. Custom middleware should solve a real recurring policy/context problem.
10. Context management must be evaluated with both quality and economics.

Final mental model:

```text
MODEL CONTEXT
   ↑
selected/compacted from:

system instructions
memory
skills
thread history
summaries
files
tool schemas
```

Next:
**Notebook 13 — Subagents**